## Glioblastoma Drug Screen

In [1]:
import sys
sys.path.append('../../../../')
import numpy as np, pandas as pd
from polygene.model.model import load_trained_model
model, tokenizer = load_trained_model("../../../../runs/gesam_polygene_run_4/")

data_path = "/media/lleger/LaCie/mit/disease_geometry/attributions/"

import os
os.listdir(data_path)
cxg_embeddings = pd.read_pickle(data_path + "glioblastoma_embeddings.pkl")
print(pd.DataFrame(cxg_embeddings[2], columns=tokenizer.phenotypic_types).value_counts("disease"),
      pd.DataFrame(cxg_embeddings[2], columns=tokenizer.phenotypic_types).value_counts("cell_type"))

study_rankings = pd.read_pickle(data_path + "../vectors/glioblastoma_study/embeddings.pkl")
study_rankings[tokenizer.phenotypic_types] = np.array(study_rankings['predictions'].tolist())
print(study_rankings.columns)

/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


disease
[normal]          50000
[glioblastoma]    15848
dtype: int64 cell_type
[oligodendrocyte_precursor_cell]    55848
[malignant_cell]                    10000
dtype: int64
Index(['embeddings', 'labels', 'predictions', 'drug', 'gradients', 'metric',
       'cell_type', 'development_stage', 'disease', 'sex', 'tissue'],
      dtype='object')


In [ ]:
# Load glioblastoma from CxG
import sys
sys.path.append('../../../../')
import numpy as np
import pandas as pd
from polygene.model.model import load_trained_model
m, tok = load_trained_model("../../../../runs/gesam_polygene_run_4/")
EMBEDDING_DIR = '/media/lleger/LaCie/mit/disease_vector/vector_data/glioblastoma_study/'
data_path = "/media/lleger/LaCie/dump/disease-vector/glioblastoma/GSE148842_RAW/"
cxg_embeddings = pd.read_pickle(EMBEDDING_DIR + "../glioblastoma_embeddings.pkl")
normal_embeddings = cxg_embeddings[0][cxg_embeddings[2][:, tok.phenotypic_types.index('disease')] == "[normal]"]
disease_embeddings = cxg_embeddings[0][cxg_embeddings[2][:, tok.phenotypic_types.index('disease')] != "[normal]"]
a, b = np.mean(normal_embeddings, axis=0), np.mean(disease_embeddings, axis=0)
# Gather new embeddings

In [ ]:
pd.Series(cxg_embeddings[2][: , 0]).value_counts()

In [ ]:
pd.Series(cxg_embeddings[2][:, tok.phenotypic_types.index('disease')]).value_counts(), disease_embeddings.shape

In [ ]:
from polygene.eval.metrics import prepare_cell
import torch
from tqdm import tqdm
import os
import scanpy as sc
embeddings, labels, predictions, drug = ([] for _ in range(4))
for path in os.listdir(EMBEDDING_DIR):
    if path.endswith('pkl'): continue
    ad = sc.read_h5ad(EMBEDDING_DIR + path)#[:1000]
    for cell in tqdm(ad):
        cell_dict = prepare_cell(cell, tok)
        cell_dict['input_ids'][np.arange(1, 1 + len(tok.phenotypic_types))] = 2
        with torch.no_grad():
            output = m(**{key: val.to(m.device).unsqueeze(0) for key, val in cell_dict.items() if key != 'str_labels'})
        encoder_output = output.hidden_states
        embeddings.append(encoder_output[0, 1 + tok.phenotypic_types.index('disease')].detach().cpu().numpy())
        labels.append(cell_dict['str_labels'][1:1 + len(tok.phenotypic_types)])
        predictions.append([tok.flattened_tokens[output.logits.argmax(dim=-1).squeeze()[1 + idx]] 
                    for idx in range(len(tok.phenotypic_types))])
    drug.extend( metadata_matrix[metadata_matrix['filename'] == path.split('_')[0]]['drug'].tolist() * len(ad))
    
df_g = pd.DataFrame({"embeddings": embeddings, "labels": labels, "predictions": predictions, "drug": drug})
df_g.to_pickle(EMBEDDING_DIR + "embeddings.pkl")

In [ ]:
# Analysis of df_g
import numpy as np
df_g = pd.read_pickle(EMBEDDING_DIR + "embeddings.pkl")
df_g['disease'] = np.array(df_g['predictions'].tolist())[:, tok.phenotypic_types.index('disease')]



v = b - a 
v /= np.dot(v,v)


s_a = np.array(df_g[(df_g['disease'] == "[normal]") & (df_g['drug'] == "treatment: none")]['embeddings'].tolist()).mean(axis=0)

#df_g = df_g[df_g['disease'] == '[glioblastoma]']

study_ranking = drug_effectiveness_rank = {
    "treatment: 0.2 uM panobinostat": 1,
    "treatment: 2.5 uM etoposide": 2,
    "treatment: 1.8 nM Ispenisib": 3,
    "treatment: 50 uM Tazemetostat": 4,
    "treatment: 50 nM RO4929097": 5,
    "treatment: 40 nM Ana-12": 6,
    "treatment: vehicle (DMSO)": 7,
    "treatment: none": 8
}
display(df_g.value_counts(['disease', 'drug']).reset_index())
centroid_study = df_g.groupby('drug').apply(lambda g: pd.Series({'centroid': np.array(g['embeddings']).mean(axis=0)})).reset_index()
centroid_study['Disease Vector Position'] = centroid_study['centroid'].apply(lambda c: (c - a) @ v)
centroid_study['Cosine Similarity of centroid vectors'] = centroid_study['centroid'].apply(lambda s: (np.dot(s-s_a, b-a))/(np.linalg.norm(b-a) * np.linalg.norm(s-s_a)))
centroid_study['Study Ranking'] = centroid_study['drug'].apply(lambda x: study_ranking.get(x,x))

#centroid_study[['drug' ,'Disease Vector Position', 'Cosine Similarity of centroid vectors', 'Study Ranking']].sort_values('Disease Vector Position')
centroid_study['drug'] = centroid_study['drug'].apply(lambda y: y.title())
centroid_study['Disease Vector Position'] = centroid_study['Disease Vector Position'].apply(lambda y: round(y,3))
centroid_study['Disease Vector Position'] = centroid_study['Disease Vector Position'].apply(lambda y: round(y,3))
centroid_study[['drug' ,'Disease Vector Position', 'Cosine Similarity of centroid vectors', 'Study Ranking']].sort_values('Cosine Similarity of centroid vectors')

In [ ]:
display(df_g.value_counts(['disease', 'drug']).reset_index())
centroid_study = df_g.groupby('drug').apply(lambda g: pd.Series({'centroid': np.array(g['embeddings']).mean(axis=0)})).reset_index()
centroid_study['Disease Vector Position'] = centroid_study['centroid'].apply(lambda c: (c - s_a) @ v)
centroid_study['Cosine Similarity of Disease Vectors'] = centroid_study['centroid'].apply(lambda s: (np.dot(s-s_a, b-a))/(np.linalg.norm(b-a) * np.linalg.norm(s-s_a)))
centroid_study['Study Ranking'] = centroid_study['drug'].apply(lambda x: study_ranking.get(x,x))

#centroid_study[['drug' ,'Disease Vector Position', 'Cosine Similarity of centroid vectors', 'Study Ranking']].sort_values('Disease Vector Position')
centroid_study['drug'] = centroid_study['drug'].apply(lambda y: ' '.join(y.split(' ')[1:][::-1]).title())
centroid_study['Cosine Similarity of Disease Vectors'] = centroid_study['Cosine Similarity of Disease Vectors'].apply(lambda y: round(y,3))
result = centroid_study[['drug' ,"Cosine Similarity of Disease Vectors", 'Study Ranking']].sort_values('Cosine Similarity of Disease Vectors')
result.columns = ['Drug', 'Cosine Similarity of Disease Vectors', 'Study Ranking']
print(result.to_latex(index=None))

In [ ]:
from sklearn.decomposition import PCA

proj = PCA(n_components=2, whiten=True, svd_solver="full")
df_cxg = pd.DataFrame({"embeddings": cxg_embeddings[0].tolist(), "disease": cxg_embeddings[2][:, 2]})
#df_cxg.sample()
#proj.fit(np.concatenate([np.array(df_g['embeddings'].tolist()), np.array(df_cxg['embeddings'].tolist())]))
proj.fit(np.concatenate([np.array(df_g['embeddings'].tolist())]))
#proj.fit(np.concatenate([np.array(df_cxg['embeddings'].tolist())]))
df_g[['x', 'y']] = proj.transform(np.array(df_g['embeddings'].tolist())).tolist()
df_cxg[['x', 'y']] = proj.transform(np.array(df_cxg['embeddings'].tolist())).tolist()
df_g = df_g[df_g['disease'].isin(['[glioblastoma]', '[normal]'])]

In [ ]:
df_g.sample()

In [ ]:
df_cxg = pd.DataFrame({"embeddings": cxg_embeddings[0].tolist(), "disease": cxg_embeddings[2][:, 2]})
df_cxg[['x', 'y']] = proj.transform(np.array(df_cxg['embeddings'].tolist())).tolist()

df_cxg.sample()

In [ ]:
SAVE = False
fontsize = 12
surround_border = 5
surround_border_y = 1
import scanpy as sc, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.patches import FancyArrowPatch
from matplotlib.legend_handler import HandlerPatch

explained_var = proj.explained_variance_ratio_
fig, ax = plt.subplots(1, 1, figsize=(9,4), dpi=100)

all_diseases = pd.concat([df_g['disease'], df_cxg['disease']]).unique().tolist()
palette = dict(zip(all_diseases, sns.color_palette('muted', n_colors=len(all_diseases))))

sns.scatterplot(data=df_g, x='x', y='y', hue=df_g['disease'],
                palette=palette, s=1, alpha=0.3, ax=ax, linewidth=0, zorder=1)
sns.scatterplot(data=df_cxg, x='x', y='y', hue=df_cxg['disease'],
                palette=palette, s=1, alpha=0.3, ax=ax, linewidth=0, zorder=1)

normal_emb_g = np.vstack(df_g[df_g['disease']=="[normal]"]['embeddings']).mean(0)
drug_vectors = {}
for drug in df_g['drug'].unique():
    print((df_g['disease']=='[glioblastoma]').sum())
    drug_emb = np.vstack(df_g[(df_g['drug']==drug) & (df_g['disease']=='[glioblastoma]')]['embeddings']).mean(0)
    drug_vectors[drug] = drug_emb - normal_emb_g

drug_colors = sns.color_palette('deep', n_colors=len(drug_vectors))
normal_umap_g = df_g[df_g['disease']=="[normal]"][['x','y']].mean().values

normal_emb_c = np.vstack(df_cxg[df_cxg['disease']=="[normal]"]['embeddings']).mean(0)
gbm_emb_c = np.vstack(df_cxg[df_cxg['disease']=="[glioblastoma]"]['embeddings']).mean(0)
gbm_vector = gbm_emb_c - normal_emb_c
cos = lambda a,b: np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

for (drug, v), c in zip(drug_vectors.items(), drug_colors):
    r = cos(v, gbm_vector)
    end = df_g[df_g['drug']==drug][['x','y']].mean().values
    ax.add_patch(FancyArrowPatch(tuple(normal_umap_g), tuple(end),
                                 arrowstyle='-|>', mutation_scale=15,
                                 color=c, linewidth=1.2, alpha=0.9, zorder=3,
                                 label=f"{drug.split(' ')[-1].title()} Disease Vector ($\\rho$={r:.2f})"))

normal_umap_c = df_cxg[df_cxg['disease']=="[normal]"][['x','y']].mean().values
gbm_umap_c = df_cxg[df_cxg['disease']=="[glioblastoma]"][['x','y']].mean().values

ax.add_patch(FancyArrowPatch(tuple(normal_umap_c), tuple(gbm_umap_c),
                             arrowstyle='-|>', mutation_scale=15,
                             color='black', linewidth=1.4, alpha=0.9,
                             zorder=3, label="Glioblastoma Disease Vector"))

def make_legend_quiver(legend, orig_handle, xdescent, ydescent, width, height, fontsize):
    return FancyArrowPatch((0, height*0.5), (width, height*0.5),
                           arrowstyle='-|>', mutation_scale=fontsize,
                           color=orig_handle.get_edgecolor() if hasattr(orig_handle,"get_edgecolor") else orig_handle.get_facecolor(),
                           linewidth=1)

ax.set_xticklabels([]); ax.set_yticklabels([])
ax.spines[['top','right']].set_linewidth(0)
ax.spines[['bottom','left']].set_linewidth(2)
ax.set_xlim(df_g['x'].quantile(.01)-surround_border, df_g['x'].quantile(.99)+surround_border)
ax.set_ylim(df_g['y'].quantile(.01)-surround_border_y, df_g['y'].quantile(.99)+surround_border_y)
ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}%)', fontsize=fontsize-2)
ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}%)', fontsize=fontsize-2)

handles, legend_labels_ = ax.get_legend_handles_labels()
legend_labels_ = [' '.join([w.capitalize() for w in x[1:-1].split('_')]).replace(' Right Ventricular','')
                  if '[' in x else x for x in legend_labels_]
handles, legend_labels_ = zip(*[(h, l) for h, l in zip(handles, legend_labels_) if "Vector" in l])

legend = ax.legend(handles, legend_labels_, frameon=True, edgecolor='none',
                   framealpha=0.8, fontsize=fontsize-2,
                   handler_map={FancyArrowPatch: HandlerPatch(patch_func=make_legend_quiver)})

for h in legend.legendHandles:
    if hasattr(h,"set_markersize"): h.set_markersize(6); h.set_alpha(1)

plt.tight_layout()
plt.show()


In [ ]:
SAVE = False
fontsize = 12
surround_border = 2
import scanpy as sc, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.patches import FancyArrowPatch
from matplotlib.legend_handler import HandlerPatch

explained_var = proj.explained_variance_ratio_
fig, ax = plt.subplots(1, 1, figsize=(6,6), dpi=200)

df_g['disease_drug'] = df_g.apply(lambda g: g['drug'].split(' ')[-1] if g['disease']=="[glioblastoma]" else g['disease'][1:-1].title(), axis=1)
sns.scatterplot(data=df_g, x='x', y='y', hue=df_g['disease_drug'],
                palette=sns.color_palette('muted', n_colors=10), s=1,
                alpha=0.3, ax=ax, linewidth=0, zorder=1)
sns.scatterplot(data=df_cxg, x='x', y='y', hue=df_cxg['disease'],
                palette=sns.color_palette('muted', n_colors=10), s=1,
                alpha=0.3, ax=ax, linewidth=0, zorder=1)

normal_emb_g = np.vstack(df_g[df_g['disease']=="[normal]"]['embeddings']).mean(0)
drug_vectors = {}
for drug in df_g['drug'].unique():
    drug_emb = np.vstack(df_g[df_g['drug']==drug]['embeddings']).mean(0)
    drug_vectors[drug] = drug_emb - normal_emb_g

normal_umap_g = df_g[df_g['disease']=="[normal]"][['x','y']].mean().values
for drug, v in drug_vectors.items():
    end = df_g[df_g['drug']==drug][['x','y']].mean().values
    ax.add_patch(FancyArrowPatch(tuple(normal_umap_g), tuple(end),
                                 arrowstyle='-|>', mutation_scale=15,
                                 color='black', linewidth=1, alpha=0.9,
                                 zorder=3, label=f"{drug.split(' ')[-1].title()} Disease Vector"))

normal_emb_c = np.vstack(df_cxg[df_cxg['disease']=="[normal]"]['embeddings']).mean(0)
gbm_emb_c = np.vstack(df_cxg[df_cxg['disease']=="[glioblastoma]"]['embeddings']).mean(0)
normal_umap_c = df_cxg[df_cxg['disease']=="[normal]"][['x','y']].mean().values
gbm_umap_c = df_cxg[df_cxg['disease']=="[glioblastoma]"][['x','y']].mean().values
ax.add_patch(FancyArrowPatch(tuple(normal_umap_c), tuple(gbm_umap_c),
                             arrowstyle='-|>', mutation_scale=15,
                             color='red', linewidth=1, alpha=0.9,
                             zorder=3, label="Glioblastoma Disease Vector"))

def make_legend_quiver(legend, orig_handle, xdescent, ydescent, width, height, fontsize):
    return FancyArrowPatch((0, height*0.5), (width, height*0.5), arrowstyle='-|>', mutation_scale=fontsize, color='black', linewidth=1)

ax.set_xticklabels([]); ax.set_yticklabels([])
ax.spines[['top','right']].set_linewidth(0)
ax.spines[['bottom','left']].set_linewidth(2)
ax.set_xlim(df_g['x'].quantile(.01)-surround_border, df_g['x'].quantile(.99)+surround_border)
ax.set_ylim(df_g['y'].quantile(.01)-surround_border, df_g['y'].quantile(.99)+surround_border)
ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}%)', fontsize=fontsize-2)
ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}%)', fontsize=fontsize-2)

handles, legend_labels_ = ax.get_legend_handles_labels()
legend_labels_ = [' '.join([w.capitalize() for w in x[1:-1].split('_')]).replace(' Right Ventricular','') if '[' in x else x for x in legend_labels_]
legend = ax.legend(handles, legend_labels_, frameon=True, edgecolor='none', framealpha=0.8, fontsize=fontsize-2, handler_map={FancyArrowPatch: HandlerPatch(patch_func=make_legend_quiver)})
for h in legend.legendHandles:
    if hasattr(h,"set_markersize"): h.set_markersize(6); h.set_alpha(1)

plt.tight_layout()
plt.show()


In [ ]:
SAVE = False
fontsize = 12
surround_border = 2
import scanpy as sc, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
explained_var = proj.explained_variance_ratio_
fig, ax = plt.subplots(1, 1, figsize=(6,6), dpi=200)

df_g['disease_drug'] = df_g.apply(lambda g: g['drug'].split(' ')[-1] if g['disease'] == "[glioblastoma]" else g['disease'][1:-1].title(), axis=1)
sns.scatterplot(data=df_g, x='x', y='y', hue=df_g['disease_drug'],
                palette=sns.color_palette('muted', n_colors=10), s=1,
                alpha=0.3, ax=ax, linewidth=0, zorder=1)

sns.scatterplot(data=df_cxg, x='x', y='y', hue=df_cxg['disease'],
                palette=sns.color_palette('muted', n_colors=10), s=1,
                alpha=0.3, ax=ax, linewidth=0, zorder=1)


ax.set_xticklabels([]); ax.set_yticklabels([])
ax.spines[['top','right']].set_linewidth(0)
ax.spines[['bottom','left']].set_linewidth(2)
ax.set_xlim(df_g['x'].quantile(.01)-surround_border, df_g['x'].quantile(.99)+surround_border)
ax.set_ylim(df_g['y'].quantile(.01)-surround_border, df_g['y'].quantile(.99)+surround_border)
ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}%)', fontsize=fontsize-2)
ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}%)', fontsize=fontsize-2)

handles, legend_labels_ = ax.get_legend_handles_labels()
legend_labels_ = [' '.join([word.capitalize() for word in x[1:-1].split('_')]).replace(' Right Ventricular', '')
                if '[' in x else x for x in legend_labels_]
#legend_labels_ = [x.replace(' ', '\n') if len(x) > 17 and "\n" not in x else x for x in legend_labels_]
legend = ax.legend(handles, legend_labels_, frameon=True, edgecolor='none', framealpha=0.8,
                fontsize=fontsize-2,)# handler_map={FancyArrowPatch: HandlerPatch(patch_func=make_legend_quiver)})
for handle in legend.legendHandles:
    if hasattr(handle, "set_markersize"): 
        handle.set_markersize(6)
        handle.set_alpha(1)

plt.tight_layout()

#if SAVE: plt.savefig(f"../figures/winding/{file_path.split('.')[0]}_{metric_name}.png", dpi=600)
plt.show()
#break

In [ ]:
centroid_study[['drug' ,'Disease Vector Position', 'Cosine Similarity of centroid vectors', 'Study Ranking']].sort_values('Disease Vector Position')
# features related to the change of state. 

In [ ]:
df_g = pd.DataFrame({"embeddings": embeddings, "labels": labels, "predictions": predictions, "drug": drug})
df_g.to_pickle(EMBEDDING_DIR + "embeddings.pkl")

In [ ]:

df_g['disease'] = np.array(df_g['predictions'].tolist())[:, tok.phenotypic_types.index('disease')]